# 07 – Disturbance Rejection

**Manufacturing context:** A CNC milling axis encounters a cutting-force disturbance mid-travel. The controller must reject this load and return to the commanded position.

Feedback control detects and corrects for unmeasured disturbances — a major advantage over open-loop operation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Editable parameters ----
# Plant
m = 1.0
c = 1.0
k = 4.0
ref = 1.0
t_end = 12.0
dt = 0.001

# PID gains
Kp = 30.0
Ki = 15.0
Kd = 8.0

# Disturbance
dist_force = -10.0    # [N] - cutting force pushing axis off target
t_dist = 5.0          # [s] - disturbance onset
dist_duration = 3.0   # [s] - how long it lasts
# -----------------------------

In [ ]:
t = np.arange(0, t_end, dt)

def simulate_with_disturbance(Kp, Ki, Kd):
    n = len(t)
    x = np.zeros(n)
    v = np.zeros(n)
    u = np.zeros(n)
    d = np.zeros(n)
    integral_e = 0.0
    prev_e = 0.0

    for i in range(1, n):
        e = ref - x[i-1]
        integral_e += e * dt
        derivative_e = (e - prev_e) / dt
        u[i] = Kp * e + Ki * integral_e + Kd * derivative_e
        prev_e = e

        # Disturbance active in window
        if t_dist <= t[i] <= t_dist + dist_duration:
            d[i] = dist_force

        a = (u[i] + d[i] - c * v[i-1] - k * x[i-1]) / m
        v[i] = v[i-1] + a * dt
        x[i] = x[i-1] + v[i] * dt

    return x, u, d

x_ctrl, u_ctrl, d = simulate_with_disturbance(Kp, Ki, Kd)
x_open, _, _ = simulate_with_disturbance(0, 0, 0)

print(f"Disturbance: {dist_force} N from t={t_dist}s to t={t_dist + dist_duration}s")
print(f"Max deviation (PID controlled) : {np.max(np.abs(x_ctrl - ref)):.4f} m")
print(f"Max deviation (uncontrolled)   : {np.max(np.abs(x_open - ref)):.4f} m")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

axes[0].plot(t, x_ctrl, label="PID controlled")
axes[0].plot(t, x_open, label="Uncontrolled", alpha=0.7)
axes[0].axhline(ref, color="k", linestyle="--", linewidth=0.8, label="Reference")
axes[0].axvspan(t_dist, t_dist + dist_duration, alpha=0.1, color="red", label="Disturbance window")
axes[0].set_ylabel("Position [m]")
axes[0].set_title("Disturbance Rejection - Cutting Force on Axis")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, u_ctrl, color="tab:orange", label="Control effort")
axes[1].set_ylabel("Control effort [N]")
axes[1].set_xlabel("Time [s]")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()

### Student Exercise

1. Double `dist_force` to -20 N. Does the controller still recover?
2. Remove the integral term (`Ki=0`). What happens to recovery after the disturbance ends?
3. In a real CNC machine, what physical phenomenon does `dist_force` represent?